# Agentic Workflow Demo: Customer Insight Agent

ReAct-style agent that orchestrates NLP tools through a reasoning loop:
`THINK -> ACT (tool call) -> OBSERVE -> ... -> ANSWER`

**7 Tools:** SEARCH, EXTRACT_ENTITIES, EXTRACT_STRUCTURED, ANALYZE_SENTIMENT, SUMMARIZE, SUMMARIZE_MULTIPLE, COMPARE. Each wraps a GCP service behind a standard interface; backends are swappable without changing agent logic.

In [1]:
import sys, os

# Set working directory to project root
os.chdir('/Users/jessepassmore/Desktop/Programming_Pizazz/nlp_fun/nlp_parsing_gcp')
sys.path.insert(0, os.getcwd())

# Load .env file (GOOGLE_API_KEY)
from dotenv import load_dotenv
load_dotenv()

if not os.environ.get('GOOGLE_API_KEY'):
    raise RuntimeError('Set GOOGLE_API_KEY in .env or export it before running this notebook.')

import pandas as pd
from IPython.display import display, Markdown, HTML

print(f'Working directory: {os.getcwd()}')
print(f'API key loaded: {os.environ["GOOGLE_API_KEY"][:15]}...')
print('Imports loaded.')

Working directory: /Users/jessepassmore/Desktop/Programming_Pizazz/nlp_fun/nlp_parsing_gcp
API key loaded: AIzaSyD1UEhY7xw...
Imports loaded.


## 1. Load Documents

Local DataFrame as search backend. In production, this becomes BigQuery: same interface, different scale.

In [2]:
from src.data.loader import load_reviews, load_support_tickets

reviews = load_reviews(max_docs=500)
tickets = load_support_tickets(max_docs=500)

# Build a unified DataFrame the agent can search
rows = []
for doc in reviews + tickets:
    rows.append({
        'id': doc.id,
        'text': doc.text,
        'source_type': doc.source_type,
        'metadata': str(doc.metadata),
    })

documents_df = pd.DataFrame(rows)
print(f'Loaded {len(documents_df)} documents ({documents_df.source_type.value_counts().to_dict()})')

/Users/jessepassmore/Desktop/Programming_Pizazz/nlp_fun/nlp_parsing_gcp/src/data/loader.py:52: DtypeWarning: Columns (0: name, 1: reviews.didPurchase) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, encoding="utf-8", on_bad_lines="skip")


Loaded 1000 documents ({'review': 500, 'support_ticket': 500})


## 2. Initialise the Agent

Gemini API key for reasoning + tool execution. Memory is local (in-memory dict) for this demo; Firestore in production.

In [3]:
from src.agent.agent import CustomerInsightAgent

agent = CustomerInsightAgent(
    api_key=os.environ['GOOGLE_API_KEY'],
    documents_df=documents_df,
    model_name='gemini-2.5-flash',
    use_firestore=False,
    max_steps=8,
)

print(f'Agent initialized with {len(agent.tools)} tools: {", ".join(agent.tools.keys())}')

Agent initialized with 7 tools: SEARCH, EXTRACT_ENTITIES, EXTRACT_STRUCTURED, ANALYZE_SENTIMENT, SUMMARIZE, SUMMARIZE_MULTIPLE, COMPARE


## 3. Helper: Reasoning Trace Visualisation

ReAct's advantage is transparency: every step is inspectable.

In [4]:
def display_agent_response(response):
    """Render the agent's reasoning trace and final answer."""
    print('=' * 70)
    print('  AGENT REASONING TRACE')
    print('=' * 70)
    for i, step in enumerate(response.steps, 1):
        print(f'\n--- Step {i} ---')
        if step.thought:
            print(f'  THOUGHT: {step.thought}')
        if step.action and step.action != 'ANSWER':
            print(f'  ACTION:  {step.action}')
            obs_preview = step.observation[:300] + '...' if len(step.observation) > 300 else step.observation
            print(f'  OBSERVE: {obs_preview}')
    print('\n' + '=' * 70)
    print('  FINAL ANSWER')
    print('=' * 70)
    print(f'\n{response.answer}')
    print(f'\n[Session: {response.session_id} | Steps: {len(response.steps)}]')

## 4. Query 1: Product Complaint Analysis

Requires search, extraction, sentiment analysis, and synthesis across multiple document types. No single API call answers this.

In [ ]:
response = agent.query(
    "What are the most common product complaints in support tickets? "
    "Find some examples and summarize the key issues."
)
display_agent_response(response)

Rate limited on gemini-2.5-flash, retrying in 10s (attempt 1/3)
Rate limited on gemini-2.5-flash, retrying in 10s (attempt 1/3)
Rate limited on gemini-2.5-flash, retrying in 20s (attempt 2/3)
Rate limited on gemini-2.5-flash, retrying in 40s (attempt 3/3)


## 5. Query 2: Sentiment Comparison

Cross-source sentiment analysis. Same session ID: the agent retains context from Query 1.

In [ ]:
response2 = agent.query(
    "How does customer sentiment differ between product reviews and support tickets? "
    "Analyze a few examples from each.",
    session_id=response.session_id,  # same session; agent remembers context
)
display_agent_response(response2)

## 6. Query 3: Entity Extraction Deep Dive

EXTRACT_ENTITIES tool: structured entity extraction with pattern identification across negative reviews.

In [ ]:
response3 = agent.query(
    "What products and brands are mentioned most frequently in negative reviews? "
    "Extract the entities and identify patterns."
)
display_agent_response(response3)

## 6b. Actor-Critic Validation

Optional critic pass: a separate Gemini call evaluates the actor's answer for completeness, factual grounding, and coherence before returning it. One extra API call; serves as both a quality gate and a production audit layer.

In [ ]:
# Initialise a critic-enabled agent: same tools, same data, plus validation
critic_agent = CustomerInsightAgent(
    api_key=os.environ['GOOGLE_API_KEY'],
    documents_df=documents_df,
    model_name='gemini-2.5-flash',
    use_firestore=False,
    max_steps=8,
    enable_critic=True,  # <-- the only difference
)

response_critic = critic_agent.query(
    "What are the top product issues in support tickets? "
    "Summarize the key themes."
)
print("=== WITH CRITIC VALIDATION ===")
display_agent_response(response_critic)

# Show the critic's structured verdict
if hasattr(critic_agent, 'last_critic_verdict') and critic_agent.last_critic_verdict:
    v = critic_agent.last_critic_verdict
    print(f'\n--- CRITIC VERDICT ---')
    print(f'  Verdict:      {v.verdict}')
    print(f'  Completeness: {v.completeness_score}/5')
    print(f'  Grounding:    {v.grounding_score}/5')
    print(f'  Coherence:    {v.coherence_score}/5')
    print(f'  Overall:      {v.overall_score}/5.0')

In [ ]:
# The CriticAgent can also be used standalone: evaluate ANY agent response
from src.agent.critic import CriticAgent

critic = CriticAgent(api_key=os.environ['GOOGLE_API_KEY'])

# Evaluate the first query's response (from the regular, non-critic agent)
verdict = critic.evaluate_agent_response(
    query="What are the most common product complaints in support tickets?",
    agent_response=response,  # response from Query 1 above
)

print('=== STANDALONE CRITIC EVALUATION ===')
print(f'  Verdict: {verdict.verdict} (overall: {verdict.overall_score}/5.0)')
print(f'  Completeness: {verdict.completeness_score}/5')
print(f'  Grounding:    {verdict.grounding_score}/5')
print(f'  Coherence:    {verdict.coherence_score}/5')
if verdict.revised_answer:
    print(f'\n  Revised answer:\n  {verdict.revised_answer[:300]}...')

## 7. Individual Tool Demos

Direct tool invocation, showing the primitives the agent orchestrates.

In [ ]:
import json

# --- SEARCH ---
search_results = agent.tools['SEARCH'].search('battery problem', source_type='review', max_results=3)
print('=== SEARCH: "battery problem" (reviews) ===')
for r in search_results:
    print(f"  [{r.get('source_type')}] {r.get('text', '')[:100]}...")

# --- EXTRACT_ENTITIES ---
if search_results:
    sample_text = search_results[0].get('text', '')
    entities = agent.tools['EXTRACT_ENTITIES'].extract_entities(sample_text)
    print(f'\n=== EXTRACT_ENTITIES ===')
    for e in entities.get('entities', [])[:5]:
        print(f"  {e['type']:12s} | {e['text']}")

In [ ]:
# --- ANALYZE_SENTIMENT ---
positive = agent.tools['ANALYZE_SENTIMENT'].analyze('This product is amazing! Best purchase ever.')
negative = agent.tools['ANALYZE_SENTIMENT'].analyze('Terrible quality. Broke after one week. Total waste of money.')

print('=== ANALYZE_SENTIMENT ===')
print(f'  Positive text: score={positive["score"]:.2f}, magnitude={positive["magnitude"]:.2f}')
print(f'  Negative text: score={negative["score"]:.2f}, magnitude={negative["magnitude"]:.2f}')

# --- SUMMARIZE ---
if search_results:
    summary = agent.tools['SUMMARIZE'].summarize(search_results[0].get('text', ''))
    print(f'\n=== SUMMARIZE ===')
    print(f'  {summary}')

## 8. Architecture Summary

```
User Query
    |
    v
+----------------------------------+
|  AGENT (Gemini reasoning loop)   |
|                                  |
|  THINK -> ACT -> OBSERVE         |
|  ...repeat until confident...    |
|  ANSWER -> CRITIC (optional)     |
+----------------------------------+
    |           |           |           |
    v           v           v           v
 SEARCH    EXTRACT     SENTIMENT   SUMMARIZE
(BigQuery/  (Gemini/    (Gemini)   (Gemini)
 DataFrame)  SpaCy)
    |           |           |           |
    v           v           v           v
+------------------------------------------+
|  MEMORY (LocalMemory / Firestore)        |
+------------------------------------------+
```

**Design decisions:** ReAct for auditable reasoning; tool abstraction decouples agent logic from backends; dual backends (local dev, GCP prod) behind same interface; session memory for multi-turn analysis; step limit (max_steps=8) prevents runaway loops.